In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dhoogla/csecicids2018/DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Bruteforce-Wedne

In [1]:
!git clone https://github.com/hosamalhmiry738-dotcom/TG-GAT-DDoS-Detection.git
%cd TG-GAT-DDoS-Detection
!pip install -r requirements.txt

Cloning into 'TG-GAT-DDoS-Detection'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 64 (delta 10), reused 33 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 98.94 KiB | 5.50 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/kaggle/working/TG-GAT-DDoS-Detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
INFO: p

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-1.13.0+cu118.html
!pip install transformers wandb pyyaml omegaconf
!pip install pandas numpy scikit-learn matplotlib seaborn plotly
!pip install pyarrow fastparquet openpyxl
!pip install captum shap lime
!pip install psutil GPUtil

print("Packages installed successfully!")

Looking in indexes: https://download.pytorch.org/whl/cu118
Looking in links: https://data.pyg.org/whl/torch-1.13.0+cu118.html
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
  Using cached torch_scatter-2.1.2.tar.gz (108 kB)
  Preparing metadata (setup.py) ... done
  Using cached torch_sparse-0.6.18.tar.gz (209 kB)
  Preparing metadata (setup.py) ... done
  Using cached torch_cluster-1.6.3.tar.gz (54 kB)
  Preparing metadata (setup.py) ... done
  Using cached torch_spline_conv-1.2.2.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=4184934 sha256=0e50c48b22200ba3dbabaa61c3460f8f782250d9babae544a6461bff6aabc523
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5


In [ ]:
# Import libraries
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import yaml
from pathlib import Path
import warnings
from tqdm import tqdm
import time
import json

# Add src to path
sys.path.append('/kaggle/working/TG-GAT-DDoS-Detection/src')

# Set up warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLibraries imported successfully!")

In [2]:
# Load configuration
from kaggle.working.TG-GAT-DDoS-Detection.src.utils.config_loader import ConfigLoader
from TG-GAT-DDoS_Detection.src.utils.wandb_logger import WandbLogger

# Load configuration
config_loader = ConfigLoader('config/default.yaml')
config = config_loader.get_config()

# Update configuration for Kaggle environment
config['hardware']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
config['training']['batch_size'] = 16 if torch.cuda.is_available() else 8  # Adjust based on GPU memory
config['training']['epochs'] = 10  # Can be increased for better results

# Create necessary directories
config_loader.create_directories()

# Print configuration
print("Configuration loaded successfully!")
print(f"\nModel Configuration:")
model_config = config['model']
for key, value in model_config.items():
    print(f"  {key}: {value}")

print(f"\nTraining Configuration:")
training_config = config['training']
for key, value in training_config.items():
    print(f"  {key}: {value}")

SyntaxError: invalid syntax (2297244171.py, line 2)